# 01 - DICOM exploration: RSNA Pneumonia sample

**Educational use only. Not a medical device.** Nothing here is a clinical finding.

This notebook explores the 600-patient sample (300 Target=1, 300 Target=0) downloaded by
`scripts/download_rsna_sample.py`. It reads the raw DICOM files, explains the header fields
and pixel handling that preprocessing will depend on, and checks the label tables for
duplicates and leakage risk. Every number shown is also written to
`reports/data_exploration.json` by the last cell.

Patient IDs are never printed: in RSNA the `patientId` is also stored inside each DICOM,
so it counts as an identifier under the project data rules.

## Setup

Loads paths and the seed from `config.yaml`. The repository root is found by walking up
from the working directory, so the notebook runs from `notebooks/` or the repo root.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import yaml
from matplotlib.patches import Rectangle
from pydicom.pixels import apply_voi_lut

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
rs = cfg["rsna_sample"]
SAMPLE_DIR = ROOT / rs["dir"]
IMAGES_DIR = SAMPLE_DIR / rs["images_subdir"]
SEED = cfg["project"]["seed"]
rng = np.random.default_rng(SEED)

manifest = pd.read_csv(SAMPLE_DIR / "sample_manifest.csv")
report = {"seed": SEED, "n_sample_patients": len(manifest)}
print(f"seed={SEED}; {len(manifest)} patients in sample manifest")

seed=42; 600 patients in sample manifest


## 1. One DICOM header

A DICOM file is a header of tagged fields plus the pixel data. The fields below decide how
pixels must be interpreted and reveal possible confounders:

- **Modality**: acquisition type. `CR` (computed radiography) or `DX` (digital X-ray).
- **PhotometricInterpretation**: whether low values are dark (`MONOCHROME2`) or bright
  (`MONOCHROME1`). Getting this wrong inverts the image a model sees.
- **Rows, Columns**: image height and width in pixels; sets resizing needs.
- **BitsStored**: bits per pixel actually used, so the valid intensity range is
  `0 .. 2**BitsStored - 1`. Normalisation must use this, not a guessed 255 or 4095.
- **PixelSpacing**: physical size of a pixel in mm (row, column). Needed to talk about
  real lesion sizes; resizing to 224x224 discards it.
- **ViewPosition**: `PA` (standing, beam back-to-front) or `AP` (often bedside, for
  sicker patients). AP films look different, so a model can learn view as a shortcut.
- **PatientSex, PatientAge**: demographics; age is a string in years. Useful for checking
  that the sample is not skewed, never as model inputs here.

In [2]:
FIELDS = ["Modality", "PhotometricInterpretation", "Rows", "Columns", "BitsStored",
          "PixelSpacing", "ViewPosition", "PatientSex", "PatientAge"]

# First positive patient in the sorted manifest; the ID itself is never printed.
example_id = manifest.loc[manifest["Target"] == 1, "patientId"].iloc[0]
ds = pydicom.dcmread(IMAGES_DIR / f"{example_id}.dcm")

example = {}
for field in FIELDS:
    value = ds.get(field)
    if field == "PixelSpacing":
        value = [float(v) for v in value]
    elif not isinstance(value, int):
        value = str(value)
    example[field] = value
    print(f"{field:<26} {value}")
report["example_dicom"] = example

Modality                   CR
PhotometricInterpretation  MONOCHROME2
Rows                       1024
Columns                    1024
BitsStored                 8
PixelSpacing               [0.168, 0.168]
ViewPosition               AP
PatientSex                 F
PatientAge                 52


## 2. Raw pixels vs display-ready pixels

`pixel_array` is the stored values, not necessarily what a radiologist sees. Two steps
turn it into a consistent image:

- **VOI LUT / windowing**: an optional lookup table or window centre/width in the header
  that maps stored values to display values. `apply_voi_lut` applies it if present and
  returns the array unchanged if not.
- **MONOCHROME1 vs MONOCHROME2**: in MONOCHROME2, 0 is black and higher values are
  brighter (bone appears white). MONOCHROME1 is the reverse, so it is inverted here.
  Without this, a dataset mixing both would show the model some images in negative.

The cell prints whether these steps changed anything for this file, rather than assuming.

In [3]:
def to_display(d: pydicom.Dataset) -> np.ndarray:
    """Apply the VOI LUT or window if present and invert MONOCHROME1 so bone is bright."""
    arr = apply_voi_lut(d.pixel_array, d).astype(np.float32)
    if d.PhotometricInterpretation == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


raw = ds.pixel_array
shown = to_display(ds)
changed = not np.array_equal(raw.astype(np.float32), shown)
has_voi = any(tag in ds for tag in ("VOILUTSequence", "WindowCenter"))
print(f"dtype={raw.dtype}, raw range=[{raw.min()}, {raw.max()}], "
      f"processed range=[{shown.min():.0f}, {shown.max():.0f}]")
print(f"VOI LUT or window tags present: {has_voi}; pixels changed: {changed}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, img, title in zip(axes, [raw, shown], ["raw pixel_array", "after VOI LUT + polarity"],
                          strict=True):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()
report["example_pixels"] = {"dtype": str(raw.dtype), "raw_min": int(raw.min()),
                            "raw_max": int(raw.max()), "has_voi_or_window": has_voi,
                            "changed_by_voi_and_polarity": changed}

dtype=uint8, raw range=[0, 255], processed range=[0, 255]
VOI LUT or window tags present: False; pixels changed: False


## 3. Labels: class balance and duplicate rows

`stage_2_train_labels.csv` has one row **per bounding box**, not per patient. A patient
with two opacities has two rows with Target=1; a negative patient has one row with empty
box columns. `stage_2_detailed_class_info.csv` splits Target=0 into "Normal" and
"No Lung Opacity / Not Normal" (abnormal for another reason).

So row counts overstate positives, and class balance must be counted per patient. These
numbers are for the **full** training labels, not the balanced sample.

In [4]:
labels = pd.read_csv(SAMPLE_DIR / rs["labels_csv"])
class_info = pd.read_csv(SAMPLE_DIR / rs["class_info_csv"])
print(f"labels rows={len(labels)}, class_info rows={len(class_info)}")

# class_info repeats per box too. Join one class row per patient so multi-box patients
# are not multiplied; validate makes pandas fail loudly if that assumption breaks.
assert class_info.groupby("patientId")["class"].nunique().max() == 1
merged = labels.merge(class_info.drop_duplicates("patientId"), on="patientId",
                      how="left", validate="many_to_one")

per_patient = merged.drop_duplicates("patientId")
rows_per_patient = merged.groupby("patientId").size()
stats = {
    "label_rows": len(merged),
    "unique_patients": int(per_patient.shape[0]),
    "patients_target_1": int((per_patient["Target"] == 1).sum()),
    "patients_target_0": int((per_patient["Target"] == 0).sum()),
    "positive_fraction": round(float(per_patient["Target"].mean()), 4),
    "class_counts": {k: int(v) for k, v in per_patient["class"].value_counts().items()},
    "max_rows_per_patient": int(rows_per_patient.max()),
}
for key, value in stats.items():
    print(f"{key:<22} {value}")
report["full_labels"] = stats
pd.crosstab(per_patient["class"], per_patient["Target"])

labels rows=30227, class_info rows=30227
label_rows             30227
unique_patients        26684
patients_target_1      6012
patients_target_0      20672
positive_fraction      0.2253
class_counts           {'No Lung Opacity / Not Normal': 11821, 'Normal': 8851, 'Lung Opacity': 6012}
max_rows_per_patient   4


Target,0,1
class,,
Lung Opacity,0,6012
No Lung Opacity / Not Normal,11821,0
Normal,8851,0


## 4. Positive cases with their bounding boxes

Nine randomly chosen (seeded) positive patients from the sample, with every annotated
opacity box drawn in red. The boxes are coarse regions marked by radiologists, not
pixel-accurate outlines, which is one reason this project uses classification plus
Grad-CAM rather than precise localisation. Titles use a case number, not the patient ID.

In [5]:
pos_ids = manifest.loc[manifest["Target"] == 1, "patientId"].to_numpy()
grid_ids = rng.choice(pos_ids, size=9, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, (ax, pid) in enumerate(zip(axes.flat, grid_ids, strict=True), start=1):
    ax.imshow(to_display(pydicom.dcmread(IMAGES_DIR / f"{pid}.dcm")), cmap="gray")
    boxes = merged.loc[merged["patientId"] == pid, ["x", "y", "width", "height"]]
    for x, y, w, h in boxes.itertuples(index=False):
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor="red", linewidth=2))
    ax.set_title(f"case {i}: {len(boxes)} box(es)")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Leakage risk

If data were split into train/validation/test **by row**, the box rows of one patient
could land in different splits. The model would then be tested on an image it trained on,
inflating every metric. Splits must therefore be made by `patientId`.

Known limitation: in RSNA each `patientId` corresponds to one image, and the dataset does
not reveal whether two IDs belong to the same real person, so that cannot be checked.

In [6]:
multi = rows_per_patient[rows_per_patient > 1]
target_by_patient = per_patient.set_index("patientId")["Target"]
sample_rows = merged[merged["patientId"].isin(manifest["patientId"])]

leakage = {
    "patients_with_multiple_rows": int(len(multi)),
    "rows_from_those_patients": int(multi.sum()),
    "multi_row_patients_all_target_1": bool((target_by_patient.loc[multi.index] == 1).all()),
    "sample_unique_patients": int(manifest["patientId"].nunique()),
    "sample_label_rows": int(len(sample_rows)),
    "sample_dicom_files": len(list(IMAGES_DIR.glob("*.dcm"))),
}
for key, value in leakage.items():
    print(f"{key:<34} {value}")
report["leakage"] = leakage

patients_with_multiple_rows        3398
rows_from_those_patients           6941
multi_row_patients_all_target_1    True
sample_unique_patients             600
sample_label_rows                  777
sample_dicom_files                 600


## 6. Image sizes and intensity ranges across the sample

Before choosing preprocessing, check that all images share a size and bit depth, and look
at how the per-image minimum, mean and maximum intensities spread. A wide spread in mean
brightness means per-image normalisation is needed; differences between PA and AP views
by class would be a shortcut the model could exploit.

In [7]:
records = []
for pid, target in manifest[["patientId", "Target"]].itertuples(index=False):
    d = pydicom.dcmread(IMAGES_DIR / f"{pid}.dcm")
    px = d.pixel_array
    records.append({"size": f"{d.Rows}x{d.Columns}", "bits": int(d.BitsStored),
                    "photometric": d.PhotometricInterpretation, "view": d.ViewPosition,
                    "target": target, "min": int(px.min()), "max": int(px.max()),
                    "mean": float(px.mean())})
img = pd.DataFrame(records)
print("sizes:", img["size"].value_counts().to_dict())
print("bits stored:", img["bits"].value_counts().to_dict())
print("photometric:", img["photometric"].value_counts().to_dict())
print(img[["min", "mean", "max"]].describe().round(1))
print(pd.crosstab(img["view"], img["target"]))

fig, ax = plt.subplots(figsize=(9, 4))
for col in ("min", "mean", "max"):
    ax.hist(img[col], bins=40, alpha=0.6, label=f"per-image {col}")
ax.set(xlabel="pixel value", ylabel="images", title="Intensity range across the sample")
ax.legend()
plt.show()

sizes: {'1024x1024': 600}
bits stored: {8: 600}
photometric: {'MONOCHROME2': 600}
         min   mean    max
count  600.0  600.0  600.0
mean     0.4  123.3  247.6
std      2.1   22.0   15.5
min      0.0   41.7  155.0
25%      0.0  109.3  247.0
50%      0.0  119.5  255.0
75%      0.0  134.6  255.0
max     20.0  188.5  255.0
target    0    1
view            
AP      112  217
PA      188   83


## Save the numbers

Writes every value collected above to `reports/data_exploration.json`, so any figure
quoted later in the README or report can be traced to this notebook's output.

In [8]:
desc = img[["min", "mean", "max"]].describe().round(2)
report["sample_images"] = {
    "sizes": {k: int(v) for k, v in img["size"].value_counts().items()},
    "bits_stored": {str(k): int(v) for k, v in img["bits"].value_counts().items()},
    "photometric": {k: int(v) for k, v in img["photometric"].value_counts().items()},
    "intensity_summary": {c: {k: float(v) for k, v in desc[c].items()} for c in desc},
    "view_by_target": {f"{v}_target_{t}": int(n) for (v, t), n in
                       img.groupby(["view", "target"]).size().items()},
}
out = ROOT / "reports" / "data_exploration.json"
out.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"wrote {out.relative_to(ROOT)} with sections: {list(report)}")

wrote reports\data_exploration.json with sections: ['seed', 'n_sample_patients', 'example_dicom', 'example_pixels', 'full_labels', 'leakage', 'sample_images']
